#  Detector de Exoplanetas — Modo Automático Continuo
### Búsqueda aleatoria en TESS + fotometría de tránsitos (BLS)

**Proyecto universitario — Pipeline automatizado**

Este notebook busca estrellas aleatorias del **TESS Input Catalog (TIC)**, descarga sus curvas de luz públicas de la NASA (misión TESS, archivo MAST) y ejecuta un pipeline completo de detección de tránsitos:

1. Elige un TIC aleatorio que no esté en el historial.
2. Descarga y limpia la curva de luz (`lightkurve`, detrending Savitzky-Golay).
3. Ejecuta **BLS (Box Least Squares)** sobre miles de períodos de prueba.
4. Busca una posible **segunda señal** (indicio de sistema multiplanetario) enmascarando el primer tránsito.
5. Clasifica la señal como sin señal / baja / media / alta confianza, y marca posibles **binarias eclipsantes**.
6. Consulta el **NASA Exoplanet Archive** (TAP) para comparar con planetas ya confirmados del mismo TIC.
7. Estima radio planetario, semieje mayor (3ª ley de Kepler), temperatura de equilibrio y densidad.
8. Guarda cada resultado en un CSV acumulativo y genera una figura de 4 paneles por estrella.
9. Se puede ejecutar **una estrella a la vez** (botón) o en **modo continuo** (bucle automático).

---
**Estructura del notebook:**
- Sección A: Instalación, imports y estilo visual
- Sección B: Configuración
- Sección C: Interfaz manual (analizar una estrella)
- Sección D: Funciones del detector (pipeline completo)
- Sección E: Cómo usarlo
- Sección F: Modo automático continuo

> ⚠️ Una señal BLS es un **candidato**, no una confirmación de planeta. Se necesitan validaciones adicionales (velocidad radial, tránsitos repetidos, descarte de binarias) para confirmar un exoplaneta real.

## A. Instalación de dependencias

Ejecutar una sola vez. `lightkurve` es la librería oficial de NASA/MAST para datos de las misiones Kepler, K2 y TESS; `astroquery` permite consultar el TIC y el NASA Exoplanet Archive.

In [1]:
# Instalar dependencias (ejecutar solo la primera vez)
import subprocess, sys

paquetes = ['lightkurve', 'astropy', 'astroquery', 'scipy', 'pandas', 'matplotlib', 'ipywidgets']
for p in paquetes:
    try:
        __import__(p.replace('-', '_'))
        print(f'✓ {p} ya instalado')
    except ImportError:
        print(f'Instalando {p}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '-q'])
        print(f'✓ {p} instalado')

✓ lightkurve ya instalado
✓ astropy ya instalado
✓ astroquery ya instalado
✓ scipy ya instalado
✓ pandas ya instalado
✓ matplotlib ya instalado
✓ ipywidgets ya instalado


In [2]:
# Imports principales
import os
import math
import warnings
from pathlib import Path
from datetime import datetime
from astropy.coordinates import SkyCoord

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import astropy.units as u
from astropy.timeseries import BoxLeastSquares
from astropy.constants import G, M_sun, R_sun

from astroquery.mast import Catalogs
from astroquery.simbad import Simbad
from astroquery.utils.tap import TapPlus

import lightkurve as lk
import ipywidgets as widgets
from IPython.display import display, FileLink, clear_output

warnings.filterwarnings("ignore")


# Estilo visual "observatorio" — igual que en detector_exoplanetas.ipynb
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'text.color': '#e6edf3',
    'grid.color': '#21262d',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'monospace',
    'figure.dpi': 110
})

print('✓ Bibliotecas cargadas y estilo visual aplicado. Python', sys.version.split()[0])


✓ Bibliotecas cargadas y estilo visual aplicado. Python 3.14.7


## B. Configuración

In [3]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

CARPETA_SALIDA = "/home/gael/Documents/CosasEscuela/ProyectoCiencias"
NOMBRE_CSV_RESULTADOS = "resultados_exoplanetas.csv"
NOMBRE_CSV_HISTORIAL = "historial_estrellas_exoplanetas.csv"

MAG_MAX = 13.0

PERIODO_MIN_DIAS = 0.5
PERIODO_MAX_DIAS = 15.0

# Antes 6000. Se baja el valor por defecto para que el análisis de cada
# estrella sea más rápido y use menos RAM de entrada; puedes subirlo desde
# la interfaz (campo "Frecuencias BLS") si tu equipo lo permite y quieres
# más resolución en el periodograma.
N_FRECUENCIAS = 3000

SNR_BAJO = 5.0
SNR_MEDIO = 8.0
SNR_ALTO = 12.0

# --- Parámetros nuevos para cuidar RAM y velocidad --------------------
# Tope de puntos que entran a BLS: si la curva combinada tiene más, se
# binea (promedia en ventanas de tiempo) antes de buscar tránsitos. Esta
# es la palanca más grande para bajar tanto tiempo de cómputo como RAM: el
# costo de BLS escala aproximadamente de forma lineal con el número de
# puntos. Con 16 GB de RAM (+ swap), 20 000 puntos es un buen equilibrio;
# bájalo (p. ej. 10000) si tu equipo sigue yendo muy justo.
LIMITE_PUNTOS_CURVA = 20000

# Tope de sectores TESS combinados por estrella. Algunas estrellas tienen
# 20-40+ sectores observados; descargarlos y apilarlos todos puede llegar
# a varios cientos de miles de puntos en memoria ANTES de binear. Se
# priorizan los sectores más recientes.
MAX_SECTORES_TESS = 10

print("Carpeta actual:", CARPETA_SALIDA)
print(f"Config: N_FRECUENCIAS={N_FRECUENCIAS} | LIMITE_PUNTOS_CURVA={LIMITE_PUNTOS_CURVA} "
      f"| MAX_SECTORES_TESS={MAX_SECTORES_TESS}")


Carpeta actual: /home/gael/Documents/CosasEscuela/ProyectoCiencias
Config: N_FRECUENCIAS=3000 | LIMITE_PUNTOS_CURVA=20000 | MAX_SECTORES_TESS=10


### B.1 Limpieza de caché de lightkurve (archivos corruptos)

`lightkurve` guarda los FITS descargados en `~/.lightkurve/cache/mastDownload/...`. Si una descarga se corta a la mitad (caída de red, cierre del notebook, `Ctrl+C` en mal momento — común en escritorio, p. ej. en CachyOS con NetworkManager reiniciándose), queda un archivo `.fits` corrupto en esa carpeta. Astropy lo detecta cada vez que se intenta volver a leerlo y lanza un error como:

```
OSError: Header missing END card... This file may be corrupt due to an interrupted download. Please remove it from your disk and try again.
```

Ese error terminaba guardándose tal cual en la columna `mensaje` del CSV. La celda siguiente revisa toda la caché **una vez al iniciar la sesión** y borra cualquier FITS corrupto que encuentre, para que no vuelva a fallar. Además, `descargar_curva_tess` (más abajo) ahora detecta este error también durante la ejecución y se autorrepara: borra el archivo dañado y reintenta la descarga.

In [4]:
# ============================================================
# LIMPIEZA DE CACHÉ DE LIGHTKURVE (archivos FITS corruptos)
# ============================================================
from astropy.io import fits as _fits_io


def limpiar_cache_lightkurve_corrupta(directorio_cache=None, verbose=True):
    """Recorre la caché local de lightkurve (por defecto
    ~/.lightkurve/cache) e intenta abrir cada .fits; si un archivo está
    corrupto (típico de una descarga interrumpida), lo elimina para que la
    próxima búsqueda lo vuelva a descargar limpio."""
    directorio_cache = Path(directorio_cache or Path.home() / ".lightkurve" / "cache")

    if not directorio_cache.exists():
        if verbose:
            print(f"No existe caché en {directorio_cache} (nada que limpiar).")
        return 0

    revisados = 0
    eliminados = 0

    for ruta in directorio_cache.rglob("*.fits"):
        revisados += 1
        try:
            with _fits_io.open(ruta, memmap=False) as hdul:
                hdul.verify("exception")
        except Exception:
            try:
                ruta.unlink()
                eliminados += 1
                if verbose:
                    print(f"🧹 Eliminado archivo corrupto: {ruta}")
            except Exception as e:
                if verbose:
                    print(f"⚠️ No se pudo eliminar {ruta}: {e}")

    if verbose:
        print(f"Revisados {revisados} archivos FITS en caché, {eliminados} corruptos eliminados.")

    return eliminados


# Se ejecuta automáticamente al correr esta celda. Puedes volver a llamarla
# manualmente en cualquier momento si ves errores de "may be corrupt".
limpiar_cache_lightkurve_corrupta()

Revisados 53112 archivos FITS en caché, 0 corruptos eliminados.


0

## C. Interfaz manual — analizar una estrella a la vez

Escribe la carpeta de salida y los nombres de los archivos, y pulsa **ANALIZAR UNA ESTRELLA** para procesar un único TIC aleatorio.

In [5]:
# ============================================================
# INTERFAZ PARA ELEGIR DÓNDE GUARDAR Y CON QUÉ NOMBRE
# ============================================================

ruta_widget = widgets.Text(
    value=CARPETA_SALIDA,
    description="Carpeta:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "90px"}
)

nombre_widget = widgets.Text(
    value=NOMBRE_CSV_RESULTADOS,
    description="CSV:",
    layout=widgets.Layout(width="70%"),
    style={"description_width": "90px"}
)

historial_widget = widgets.Text(
    value=NOMBRE_CSV_HISTORIAL,
    description="Historial:",
    layout=widgets.Layout(width="70%"),
    style={"description_width": "90px"}
)

frecuencias_widget = widgets.IntText(
    value=N_FRECUENCIAS,
    description="Frecuencias BLS:",
    min=100,
    layout=widgets.Layout(width="320px"),
    style={"description_width": "130px"}
)

analizar_btn = widgets.Button(
    description="ANALIZAR UNA ESTRELLA",
    button_style="success",
    icon="search",
    layout=widgets.Layout(width="300px", height="45px")
)

salida_ui = widgets.Output()

display(
    ruta_widget,
    nombre_widget,
    historial_widget,
    frecuencias_widget,
    analizar_btn,
    salida_ui
)


Text(value='/home/gael/Documents/CosasEscuela/ProyectoCiencias', description='Carpeta:', layout=Layout(width='…

Text(value='resultados_exoplanetas.csv', description='CSV:', layout=Layout(width='70%'), style=TextStyle(descr…

Text(value='historial_estrellas_exoplanetas.csv', description='Historial:', layout=Layout(width='70%'), style=…

IntText(value=3000, description='Frecuencias BLS:', layout=Layout(width='320px'), style=DescriptionStyle(descr…

Button(button_style='success', description='ANALIZAR UNA ESTRELLA', icon='search', layout=Layout(height='45px'…

Output()

## D. Funciones del detector (pipeline completo)

Incluye: búsqueda de TIC aleatorios, descarga y limpieza de curvas, BLS, búsqueda de segunda señal, clasificación, consulta al NASA Exoplanet Archive, estimación de parámetros físicos y la **figura de 4 paneles estilo observatorio** (curva completa, zoom al tránsito, periodograma con armónicos y curva plegada binada), igual que en `detector_exoplanetas.ipynb`.

In [6]:
# ============================================================
# FUNCIONES DEL DETECTOR  (versión mejorada)
# ============================================================
#
# Esta celda mantiene EXACTAMENTE el mismo formato de salida (las mismas
# columnas, en el mismo orden, con el mismo significado) que la versión
# original: COLUMNAS_RESULTADOS no cambia y procesar_una_estrella() sigue
# devolviendo (fila, csv_path, hist_path). Todo lo que cambia es la
# CALIDAD del análisis que rellena esas columnas:
#
#   1. Selección de curvas TESS: si un mismo sector tiene más de un
#      pipeline (SPOC/TESS-SPOC/QLP), ya no se mezclan todos —se escoge
#      uno solo por sector, con prioridad SPOC > TESS-SPOC > QLP— para
#      no duplicar/objetar tramos de datos al hacer stitch().
#   2. Limpieza de la curva: sigma-clipping ASIMÉTRICO para no borrar los
#      puntos más profundos de un tránsito real como si fueran outliers.
#   3. BLS: rejilla de períodos NO uniforme (bls.autoperiod), que evita
#      perder picos estrechos a períodos cortos —un fallo típico de usar
#      linspace—, y rejilla de duraciones acotada por la 3ª ley de Kepler
#      cuando se conoce el radio/masa de la estrella.
#   4. Métrica adicional interna (SDE, Signal Detection Efficiency) para
#      exigir que el pico del periodograma sea significativo y no solo
#      tenga buen SNR "de libro".
#   5. Test odd-even y búsqueda de eclipse secundario en fase 0.5: dos
#      técnicas estándar para distinguir tránsitos planetarios reales de
#      binarias eclipsantes, que ahora informan `posible_binaria_eclipsante`
#      y `confianza` (columnas que YA existían).
#   6. Se descartan como candidato principal los períodos muy próximos a
#      alias instrumentales típicos de TESS (1 día, 2 días, ~13.7 días).
#   7. Selección de estrella: entre los candidatos aleatorios, se prefieren
#      los que sí tienen radio y masa catalogados en el TIC, porque son los
#      únicos para los que se pueden estimar radio planetario/semieje/T_eq.
#   8. Reintentos de descarga con backoff con jitter (antes reintentaba
#      inmediatamente).
#
# Además, esta versión consume bastante menos RAM (y disco) en sesiones
# largas de modo automático:
#   9. Ya NO se relee el CSV completo en cada estrella para añadirle una
#      fila: se abre en modo 'append' y se escribe solo la fila nueva.
#      Antes, cada llamada cargaba en memoria TODO el historial acumulado
#      (como DataFrame) solo para pegarle una fila y reescribirlo entero;
#      con miles de estrellas analizadas eso significa cargar en RAM (y
#      volver a escribir en disco) un archivo cada vez más grande, en
#      cada estrella.
#  10. cargar_ids_usados ya no carga las 35 columnas del historial, solo
#      la columna tic_id (que es lo único que necesita para no repetir
#      estrella).
#  11. Los archivos FITS descargados de cada estrella se guardan en una
#      carpeta temporal que se borra en cuanto se terminan de leer, en
#      vez de acumularse para siempre en el caché de lightkurve
#      (~/.lightkurve/cache). El pipeline nunca vuelve a analizar el
#      mismo TIC (lo evita cargar_ids_usados), así que no hay ningún
#      beneficio en conservar ese caché entre estrellas -solo va
#      llenando el disco durante una sesión larga, lo que además golpea
#      al swap.
#  12. Las figuras, en modo automático, ya NO se muestran en línea (solo
#      se guardan como PNG en disco): mostrarlas en el notebook las deja
#      incrustadas como imagen en la salida de la celda, y en un bucle
#      que corre indefinidamente eso hace crecer la memoria del kernel y
#      del navegador sin límite. En el modo manual (botón "Analizar")
#      se siguen mostrando igual que antes.
#
# Nada de esto añade columnas nuevas al CSV ni cambia su formato.

import re
import time
import random
import gc

# Estos dos parámetros viven normalmente en la celda de CONFIGURACIÓN. Si
# se ejecuta esta celda con una versión antigua de esa celda (que no los
# define), se usa aquí un valor por defecto en vez de fallar con
# NameError.
try:
    MAX_SECTORES_TESS
except NameError:
    MAX_SECTORES_TESS = 10  # tope de sectores TESS combinados por estrella

try:
    LIMITE_PUNTOS_CURVA
except NameError:
    LIMITE_PUNTOS_CURVA = 20000  # tope de puntos que entran a BLS (se binea si hay más)

import shutil
import tempfile

# Paleta de colores estilo "observatorio" (igual que detector_exoplanetas.ipynb)
PALETTE = {
    "bg": "#0d1117",
    "grid": "#21262d",
    "text": "#e6edf3",
    "muted": "#8b949e",
    "border": "#30363d",
    "blue": "#58a6ff",
    "red": "#f85149",
    "green": "#3fb950",
    "orange": "#f0883e",
    "gold": "#ffa657",
    "white": "#f0f6fc",
}


# COLUMNAS_RESULTADOS: SIN CAMBIOS respecto a la versión original. Este es
# el contrato de formato del CSV y no se toca.
COLUMNAS_RESULTADOS = [
    "tic_id",
    "nombre_estrella",
    "ra_deg",
    "dec_deg",
    "tmag",
    "teff_K",
    "radio_estrella_Rsun",
    "masa_estrella_Msun",
    "sectores_tess",
    "observacion_tess_inicio",
    "observacion_tess_fin",
    "fecha_hora_analisis",
    "datos_tess_disponibles",
    "num_puntos_curva",
    "periodo_detectado_dias",
    "snr_bls",
    "profundidad_fraccion",
    "profundidad_pct",
    "confianza",
    "anomalia_en_la_luz",
    "posible_transito",
    "posible_binaria_eclipsante",
    "posible_segunda_senal",
    "periodo_segunda_senal_dias",
    "snr_segunda_senal",
    "planeta_confirmado_en_archive",
    "planetas_conocidos",
    "periodo_publicado_mas_cercano_dias",
    "error_periodo_porcentual",
    "radio_planeta_Rearth",
    "radio_planeta_Rjup",
    "semieje_mayor_AU",
    "temperatura_equilibrio_K",
    "densidad_g_cm3",
    "estado",
    "mensaje"
]


def valor_seguro(row, columna, default=np.nan):
    try:
        valor = row[columna]
        if np.ma.is_masked(valor):
            return default
        if valor is None:
            return default
        return valor
    except Exception:
        return default


def a_float(valor, default=np.nan):
    try:
        if valor is None or np.ma.is_masked(valor):
            return default
        x = float(valor)
        return x if np.isfinite(x) else default
    except Exception:
        return default


def leer_csv_seguro(path, columnas=None):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(columns=columnas or [])
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame(columns=columnas or [])


def guardar_csv(path, df):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def cargar_ids_usados(path_historial):
    """Carga solo los tic_id ya analizados, sin traer a memoria las otras
    34 columnas del historial (mensajes, listas de planetas, etc.), que
    no hacen falta para esta comprobación. Con un historial de miles de
    filas esto evita cargar en RAM un DataFrame mucho más pesado de lo
    necesario en cada estrella nueva."""
    path = Path(path_historial)
    if not path.exists():
        return set()
    try:
        ids = pd.read_csv(path, usecols=["tic_id"])["tic_id"]
    except Exception:
        # Archivo vacío, corrupto o todavía sin esa columna: se trata
        # como "sin historial" en vez de fallar.
        return set()
    ids = pd.to_numeric(ids, errors="coerce").dropna().astype("int64")
    return set(ids.tolist())


def _escribir_fila_csv(path, fila):
    """Añade una fila al CSV en modo 'append', sin releer ni reescribir
    el archivo completo cada vez.

    La versión anterior cargaba el CSV entero en un DataFrame en CADA
    llamada, le concatenaba una fila y volvía a escribirlo desde cero;
    con un historial que crece a lo largo de una sesión larga, eso es
    releer y reescribir en disco (y mantener en RAM) un archivo cada vez
    más grande por cada estrella analizada. Aquí simplemente se abre el
    archivo en modo 'a' (append) y se escribe solo la fila nueva.

    El formato resultante del CSV es idéntico: mismas columnas, mismo
    orden, mismo separador y comillas de pandas. El BOM UTF-8 (utf-8-sig)
    solo se escribe una vez, al crear el archivo por primera vez -escribir
    un BOM en mitad de un CSV con cada 'append' lo corrompería-.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    nueva = pd.DataFrame([fila])
    for col in COLUMNAS_RESULTADOS:
        if col not in nueva.columns:
            nueva[col] = np.nan
    nueva = nueva[COLUMNAS_RESULTADOS]

    existe = path.exists() and path.stat().st_size > 0

    if not existe:
        nueva.to_csv(path, index=False, encoding="utf-8-sig", mode="w")
    else:
        nueva.to_csv(path, index=False, header=False, encoding="utf-8", mode="a")


def append_resultado(path_csv, fila):
    _escribir_fila_csv(path_csv, fila)


def append_historial(path_historial, fila):
    _escribir_fila_csv(path_historial, fila)


def coordenadas_aleatorias():
    rng = np.random.default_rng()
    ra = float(rng.uniform(0, 360))
    sin_dec = float(rng.uniform(-1, 1))
    dec = math.degrees(math.asin(sin_dec))
    return SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")


def buscar_estrellas_nuevas(ids_usados, max_intentos=30, radio_deg=0.15):
    """Busca una estrella aleatoria del TIC que aún no figure en el historial.

    Mejora: entre los candidatos válidos de la misma consulta, se prefieren
    los que además tienen radio Y masa catalogados en el TIC. Sin esos dos
    valores el pipeline no puede estimar radio planetario, semieje mayor,
    temperatura de equilibrio ni densidad (quedan como NaN igualmente en
    esas columnas, como antes), así que priorizar estrellas con parámetros
    conocidos aumenta la fracción de resultados con datos físicos
    completos sin cambiar ninguna columna del CSV. Si en una consulta
    ninguna estrella tiene esos parámetros, simplemente se sigue eligiendo
    al azar entre todos los candidatos, igual que antes.
    """
    for intento in range(1, max_intentos + 1):
        coord = coordenadas_aleatorias()

        try:
            tabla = Catalogs.query_region(
                coord,
                radius=radio_deg,
                catalog="TIC"
            )
        except Exception as e:
            print(f"Intento {intento}: fallo consultando MAST -> {e}")
            continue

        if tabla is None or len(tabla) == 0:
            continue

        candidatos = []
        for row in tabla:
            tic_id = valor_seguro(row, "ID", None)
            tmag = a_float(valor_seguro(row, "Tmag", np.nan))

            if tic_id is None or not np.isfinite(tmag):
                continue

            try:
                tic_id = int(tic_id)
            except Exception:
                continue

            if tic_id in ids_usados or tmag > MAG_MAX:
                continue

            obj_type = str(valor_seguro(row, "objType", ""))
            if obj_type and "STAR" not in obj_type.upper():
                continue

            candidatos.append(row)

        if candidatos:
            con_parametros = [
                r for r in candidatos
                if np.isfinite(a_float(valor_seguro(r, "rad", np.nan)))
                and np.isfinite(a_float(valor_seguro(r, "mass", np.nan)))
            ]
            pool = con_parametros if con_parametros else candidatos
            return pool[int(np.random.randint(0, len(pool)))]

    raise RuntimeError(
        "No se pudo encontrar una estrella nueva. "
        "Prueba de nuevo o aumenta MAG_MAX."
    )


# ============================================================
# RESOLUCIÓN DE NOMBRE DE ESTRELLA (SIMBAD)
# ============================================================
# Un TIC es solo un identificador de catálogo (TESS Input Catalog),
# no un nombre propio. Muchas estrellas TIC tienen además un nombre
# más conocido (Bayer/Flamsteed, HD, HIP, TYC, Gaia DR3, TOI, etc.)
# registrado en SIMBAD. Esta función intenta resolverlo:
#   1) Busca directamente por "TIC <id>" en SIMBAD.
#   2) Si falla (SIMBAD no tiene cargados todos los TIC), hace un
#      cone search por coordenadas (ra_deg, dec_deg) y toma el
#      objeto más cercano.
#   3) Si nada funciona, devuelve "TIC <id>" como antes.

PREFIJOS_NOMBRE_SIMBAD = [
    "NAME ", "* ", "V* ", "TOI-", "HD ", "HIP ", "HR ",
    "TYC ", "BD", "GJ ", "Gaia DR3 ", "2MASS "
]


def _elegir_mejor_nombre_simbad(lista_ids):
    """De una lista de identificadores SIMBAD, elige el más 'humano'
    según el orden de preferencia en PREFIJOS_NOMBRE_SIMBAD."""
    for prefijo in PREFIJOS_NOMBRE_SIMBAD:
        for nombre in lista_ids:
            if nombre.startswith(prefijo):
                return nombre.replace("NAME ", "").strip()
    return None


def resolver_nombre_estrella(tic_id, ra_deg=None, dec_deg=None, radio_arcsec=5.0):
    """Devuelve un nombre alternativo (más legible) para un TIC dado,
    o 'TIC <id>' si SIMBAD no tiene nada mejor."""
    fallback = f"TIC {tic_id}"

    # 1) Intento directo por identificador TIC
    try:
        tabla = Simbad.query_objectids(f"TIC {tic_id}")
        if tabla is not None and len(tabla) > 0:
            col = "ID" if "ID" in tabla.colnames else tabla.colnames[0]
            mejor = _elegir_mejor_nombre_simbad([str(x) for x in tabla[col]])
            if mejor:
                return mejor
    except Exception:
        pass

    # 2) Fallback: cone search por coordenadas
    if ra_deg is not None and dec_deg is not None and np.isfinite(ra_deg) and np.isfinite(dec_deg):
        try:
            coord = SkyCoord(ra_deg, dec_deg, unit="deg")
            reg = Simbad.query_region(coord, radius=radio_arcsec * u.arcsec)
            if reg is not None and len(reg) > 0:
                col_id = "MAIN_ID" if "MAIN_ID" in reg.colnames else reg.colnames[0]
                principal = str(reg[col_id][0])
                ids = Simbad.query_objectids(principal)
                if ids is not None and len(ids) > 0:
                    col = "ID" if "ID" in ids.colnames else ids.colnames[0]
                    mejor = _elegir_mejor_nombre_simbad([str(x) for x in ids[col]])
                    if mejor:
                        return mejor
                return principal
        except Exception:
            pass

    return fallback


def obtener_datos_tic(row):
    tic_id = int(valor_seguro(row, "ID"))
    ra_deg = a_float(valor_seguro(row, "ra"))
    dec_deg = a_float(valor_seguro(row, "dec"))
    nombre_estrella = resolver_nombre_estrella(tic_id, ra_deg=ra_deg, dec_deg=dec_deg)
    return {
        "tic_id": tic_id,
        "nombre_estrella": nombre_estrella,
        "ra_deg": ra_deg,
        "dec_deg": dec_deg,
        "tmag": a_float(valor_seguro(row, "Tmag")),
        "teff_K": a_float(valor_seguro(row, "Teff")),
        "radio_estrella_Rsun": a_float(valor_seguro(row, "rad")),
        "masa_estrella_Msun": a_float(valor_seguro(row, "mass")),
    }


# Pipelines TESS que lightkurve sabe leer de forma fiable (tienen un lector
# dedicado). Productos HLSP "comunitarios" como "t16" o "tars" no tienen
# lector propio: lightkurve cae a un parser 'generic' que casi siempre
# falla, y el error que lanza ("this file may be corrupt due to an
# interrupted download") es engañoso -no es un problema de descarga, es que
# ese pipeline no está soportado-. Por eso los excluimos desde la búsqueda
# en vez de intentar descargarlos.
AUTORES_TESS_SOPORTADOS = ["SPOC", "TESS-SPOC", "QLP"]

# Orden de preferencia cuando un mismo sector tiene más de un pipeline
# disponible: SPOC (fotometría de apertura calibrada, la más precisa) >
# TESS-SPOC (mismo pipeline reprocesado para FFI) > QLP (fotometría rápida
# de menor precisión, pensada para cobertura masiva).
PRIORIDAD_AUTOR = {"SPOC": 0, "TESS-SPOC": 1, "QLP": 2}


def _elegir_mejor_por_sector(search):
    """Si un mismo sector TESS tiene curvas de más de un pipeline (p. ej.
    SPOC y QLP a la vez), lightkurve las trataría como observaciones
    independientes y el stitch() acabaría mezclando dos versiones
    distintas del mismo tramo de datos: puntos duplicados, con
    calibraciones distintas, que solo añaden ruido y pueden diluir o
    distorsionar un tránsito real. Aquí se deja un único producto por
    sector, priorizando SPOC > TESS-SPOC > QLP.
    """
    try:
        tabla = search.table
        if "sequence_number" in tabla.colnames:
            sectores = np.asarray(tabla["sequence_number"])
        elif "sector" in tabla.colnames:
            sectores = np.asarray(tabla["sector"])
        else:
            return search
        autores = np.asarray(search.author, dtype=str)
    except Exception:
        return search

    if len(sectores) != len(autores) or len(sectores) == 0:
        return search

    lista_sectores = sorted(set(sectores.tolist()))

    # Además de quedarnos con un único pipeline por sector, se limita el
    # número TOTAL de sectores combinados a MAX_SECTORES_TESS. Una estrella
    # con muchísimos sectores (algunas tienen 20-40+ en TESS extendida)
    # puede acumular cientos de miles de puntos a 2 minutos de cadencia, lo
    # que dispara tanto la RAM de la descarga/stitch como el tiempo de BLS
    # más adelante. Se priorizan los sectores más recientes (números de
    # sector más altos), que además suelen tener mejor calibración.
    if len(lista_sectores) > MAX_SECTORES_TESS:
        lista_sectores = sorted(lista_sectores)[-MAX_SECTORES_TESS:]

    indices_elegidos = []
    for sector in lista_sectores:
        idxs = np.where(sectores == sector)[0]
        mejor_idx = min(idxs, key=lambda i: PRIORIDAD_AUTOR.get(autores[i], 99))
        indices_elegidos.append(int(mejor_idx))

    try:
        return search[sorted(indices_elegidos)]
    except Exception:
        return search


def descargar_curva_tess(tic_id, max_reintentos=3):
    """Descarga las curvas de luz TESS disponibles para el TIC, limitándose
    a los pipelines soportados por lightkurve (ver AUTORES_TESS_SOPORTADOS)
    y a un único pipeline por sector (ver _elegir_mejor_por_sector).

    Los FITS se descargan a una carpeta TEMPORAL propia de esta estrella
    (no al caché por defecto de lightkurve, que crece para siempre en
    ~/.lightkurve/cache). En cuanto se han leído en memoria -o si la
    descarga falla del todo- esa carpeta se borra, pase lo que pase
    (try/finally). El pipeline nunca vuelve a analizar el mismo TIC (lo
    evita cargar_ids_usados), así que no hay ningún beneficio real en
    conservar esos FITS entre estrellas: solo van llenando el disco en
    una sesión larga de modo automático, lo que además termina golpeando
    al swap y ralentizando todo el sistema.

    Si una descarga se corta a medias (red inestable), astropy/lightkurve
    lanzan un error avisando que el FITS "may be corrupt". En ese caso se
    vacía la carpeta temporal y se reintenta hasta `max_reintentos` veces,
    con una espera creciente (backoff con jitter) entre intentos.
    """
    search_completo = lk.search_lightcurve(f"TIC {tic_id}", mission="TESS")

    if len(search_completo) == 0:
        return None, search_completo

    try:
        autores = np.asarray(search_completo.author, dtype=str)
        mascara_soportados = np.isin(autores, AUTORES_TESS_SOPORTADOS)
        search = search_completo[mascara_soportados]
    except Exception:
        # Si por lo que sea no se puede filtrar (cambio de API, etc.),
        # seguimos con la búsqueda completa en vez de romper el pipeline.
        search = search_completo

    if len(search) == 0:
        # Solo había pipelines no soportados: se trata como "sin datos
        # utilizables", no como error.
        return None, search_completo

    search = _elegir_mejor_por_sector(search)

    tmp_dir = tempfile.mkdtemp(prefix=f"tess_tic{tic_id}_")

    try:
        collection = None
        ultimo_error = None

        for intento in range(1, max_reintentos + 1):
            try:
                collection = search.download_all(download_dir=tmp_dir)
                ultimo_error = None
                break
            except Exception as e:
                ultimo_error = e
                mensaje = str(e)
                parece_corrupto = (
                    "may be corrupt" in mensaje
                    or "interrupted download" in mensaje
                    or "Empty or corrupt FITS file" in mensaje
                )

                if parece_corrupto and intento < max_reintentos:
                    shutil.rmtree(tmp_dir, ignore_errors=True)
                    Path(tmp_dir).mkdir(parents=True, exist_ok=True)
                    espera = min(30, 2 ** intento) + random.uniform(0, 1.5)
                    print(f"↻ Reintentando descarga de TIC {tic_id} "
                          f"(intento {intento + 1}/{max_reintentos}) en {espera:.1f}s...")
                    time.sleep(espera)
                    continue

                # Error no relacionado con una descarga corrupta, o ya sin
                # más reintentos disponibles: se relanza para que quede
                # registrado en la columna 'mensaje' del CSV.
                break

        if ultimo_error is not None:
            raise ultimo_error

        if collection is None or len(collection) == 0:
            return None, search

        # Distintos pipelines (SPOC, QLP, TESS-SPOC...) usan columnas de
        # flujo distintas; nos quedamos solo con las curvas que sí tienen
        # flujo válido antes de combinarlas, para evitar errores de
        # stitch() por columnas incompatibles.
        curvas_validas = [
            c for c in collection
            if c is not None and np.any(np.isfinite(np.asarray(c.flux.value, dtype=float)))
        ]
        if not curvas_validas:
            return None, search

        if len(curvas_validas) == 1:
            lc = curvas_validas[0]
        else:
            # stitch() normaliza cada curva por separado antes de
            # concatenar (comportamiento por defecto de lightkurve), lo
            # que evita saltos de nivel entre sectores con baselines de
            # flujo distintos.
            lc = lk.LightCurveCollection(curvas_validas).stitch()

        # A partir de aquí toda la fotometría que hace falta ya vive en
        # `lc` (en memoria); los FITS de disco ya no se necesitan.
        return lc, search

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


def reducir_puntos_curva(lc, limite_puntos=None, bin_maximo_minutos=10):
    """Reduce el número de puntos de la curva bineando (promediando en
    ventanas de tiempo) si hay muchos más puntos de los necesarios.

    Esta es la palanca más grande para bajar tanto RAM como tiempo de
    cómputo: BLS recorre todos los puntos de la curva para cada período y
    duración de prueba, así que el costo escala aproximadamente de forma
    lineal con el número de puntos. Una estrella con varios sectores TESS
    a cadencia de 2 minutos puede fácilmente tener 50 000-150 000 puntos;
    bajarla a ~20 000 sin perder apenas sensibilidad acelera BLS varias
    veces y reduce la memoria que ocupan los arrays en la misma proporción.

    El tamaño del bin nunca supera bin_maximo_minutos, para no diluir
    tránsitos cortos: la mayoría de tránsitos de período corto duran entre
    1 y 4 horas, así que un bin de 10 minutos deja de sobra 6+ puntos
    dentro incluso del tránsito más corto razonable.
    """
    if limite_puntos is None:
        limite_puntos = LIMITE_PUNTOS_CURVA

    n = len(lc)
    if n <= limite_puntos:
        return lc

    try:
        duracion_total = float(lc.time.value.max() - lc.time.value.min())
        if duracion_total <= 0:
            return lc

        bin_dias = duracion_total / limite_puntos
        bin_dias = min(bin_dias, bin_maximo_minutos / 1440.0)

        if bin_dias <= 0:
            return lc

        binned = lc.bin(time_bin_size=bin_dias * u.day)
        binned = binned.remove_nans()

        if len(binned) < 300:
            # El binning dejó muy pocos puntos (curva muy corta o con
            # muchos huecos); mejor quedarse con la curva original.
            return lc

        return binned
    except Exception:
        return lc


def preparar_curva(lc):
    """Limpia y aplana la curva de luz.

    Mejora clave: el sigma-clipping ahora es ASIMÉTRICO. Un tránsito real
    es una caída de flujo; un clipping simétrico a 5-sigma puede borrar
    justo los puntos más profundos de un tránsito real (o de una binaria
    eclipsante) tratándolos como si fueran outliers, cortándole la cabeza
    a la propia señal que se quiere detectar. Se sube el límite inferior
    (sigma_lower) para proteger esas caídas y se deja estricto el límite
    superior (sigma_upper), donde sí suelen estar los rayos cósmicos y
    otros artefactos instrumentales positivos.
    """
    lc = lc.remove_nans()

    if len(lc) < 300:
        raise ValueError("La curva tiene muy pocos puntos.")

    try:
        lc = lc.remove_outliers(sigma_lower=20, sigma_upper=5)
    except TypeError:
        # Versiones muy antiguas de lightkurve sin sigma_lower/sigma_upper.
        lc = lc.remove_outliers(sigma=5)

    ventana = min(601, len(lc) - 1)
    if ventana % 2 == 0:
        ventana -= 1
    ventana = max(101, ventana)

    try:
        flat = lc.normalize().flatten(
            window_length=ventana,
            polyorder=2,
            break_tolerance=5
        )
    except Exception:
        flat = lc.normalize().flatten(
            window_length=ventana,
            polyorder=2
        )

    flat = flat.remove_nans()

    # Reducir puntos DESPUÉS de aplanar: así el binning promedia flujo ya
    # corregido de tendencias de largo plazo, en vez de mezclar niveles de
    # flujo crudo distintos dentro de un mismo bin.
    return reducir_puntos_curva(flat)


# Períodos típicos de sistemáticas instrumentales de TESS: ~1 día (alias
# de la rotación terrestre / iluminación solar en la reducción de datos) y
# ~13.7 días (período orbital de la nave; los momentum dumps y el
# downlink de datos dejan huecos/discontinuidades periódicas a ese
# ritmo). Un "candidato" cuyo período cae justo ahí es mucho más probable
# que sea sistemática que un tránsito real.
PERIODOS_INSTRUMENTALES_DIAS = [1.0, 13.7]
TOLERANCIA_PERIODO_INSTRUMENTAL = 0.02


def _es_periodo_instrumental(periodo):
    if not np.isfinite(periodo) or periodo <= 0:
        return False
    for base in PERIODOS_INSTRUMENTALES_DIAS:
        for arm in (0.5, 1, 2, 3):
            p_ref = base * arm
            if p_ref <= 0:
                continue
            if abs(periodo - p_ref) / p_ref < TOLERANCIA_PERIODO_INSTRUMENTAL:
                return True
    return False


def estimar_duracion_maxima_fisica(periodo_dias, rstar_rsun, mstar_msun):
    """Duración máxima físicamente plausible de un tránsito central, vía
    3ª ley de Kepler: T_max ≈ (P / π) · (R* / a). Cuando se conocen el
    radio y la masa estelar, esto da una rejilla de duraciones de tránsito
    mucho más ajustada que una fracción arbitraria del período, evitando
    que BLS pierda potencia probando duraciones que ninguna geometría real
    podría producir a ese período."""
    if not (
        np.isfinite(rstar_rsun) and rstar_rsun > 0
        and np.isfinite(mstar_msun) and mstar_msun > 0
        and np.isfinite(periodo_dias) and periodo_dias > 0
    ):
        return None
    try:
        P = periodo_dias * u.day
        M = mstar_msun * M_sun
        a = ((G * M * P ** 2) / (4 * np.pi ** 2)) ** (1 / 3)
        rs = rstar_rsun * R_sun
        t_max = (P / np.pi) * (rs / a).decompose().value
        valor = float(t_max.to(u.day).value)
        return valor if np.isfinite(valor) and valor > 0 else None
    except Exception:
        return None


def ejecutar_bls(lc, n_frecuencias=6000, rstar_rsun=np.nan, mstar_msun=np.nan):
    """Ejecuta BLS sobre la curva de luz.

    Dos mejoras respecto a la versión anterior:
    - Rejilla de PERÍODOS no uniforme (bls.autoperiod, uniforme en
      frecuencia): un linspace uniforme en período sobremuestrea los
      períodos largos y submuestrea los cortos, y puede saltarse picos
      estrechos de tránsitos de período corto entre dos puntos
      consecutivos de la rejilla. Si autoperiod devolviera más puntos de
      los pedidos (n_frecuencias), se diezma conservando la forma no
      uniforme en vez de recortar el rango.
    - Rejilla de DURACIONES acotada con la 3ª ley de Kepler cuando se
      conoce el radio/masa de la estrella (ver
      estimar_duracion_maxima_fisica), en vez de una fracción arbitraria
      del período máximo.
    """
    time_days = np.asarray(lc.time.value, dtype=float)
    flux = np.asarray(lc.flux.value, dtype=float)

    duracion_base = time_days.max() - time_days.min()
    pmax = min(
        PERIODO_MAX_DIAS,
        max(PERIODO_MIN_DIAS + 0.05, duracion_base / 2)
    )

    if pmax <= PERIODO_MIN_DIAS:
        raise ValueError("La ventana temporal no permite buscar períodos suficientes.")

    dur_fisica = estimar_duracion_maxima_fisica(pmax, rstar_rsun, mstar_msun)
    if dur_fisica is not None:
        # Margen de seguridad (excentricidad orbital, incertidumbre en
        # R*/M* del catálogo) sobre la duración máxima "de libro".
        max_dur = dur_fisica * 1.3
    else:
        max_dur = max(0.02, pmax * 0.08)

    # La duración máxima probada SIEMPRE debe ser menor que el período
    # mínimo probado (si no, BoxLeastSquares lanza ValueError: "The
    # maximum transit duration must be shorter than the minimum period").
    max_dur = min(max_dur, PERIODO_MIN_DIAS * 0.9)
    min_dur = min(0.01, max_dur * 0.3)
    # 9 duraciones de prueba (antes 12): al estar acotadas físicamente por
    # Kepler cuando se conoce R*/M*, no hacen falta tantas para cubrir bien
    # el rango plausible, y cada duración de menos es ~8% menos cómputo.
    durations = np.linspace(min_dur, max_dur, 9)

    bls = BoxLeastSquares(time_days, flux)

    try:
        periods = bls.autoperiod(
            durations,
            minimum_period=PERIODO_MIN_DIAS,
            maximum_period=pmax,
            minimum_n_transit=2,
            frequency_factor=1.0
        )
        periods = np.asarray(periods, dtype=float)
        if len(periods) > n_frecuencias:
            paso = max(1, len(periods) // int(n_frecuencias))
            periods = periods[::paso]
        periods = np.unique(np.sort(periods))
        if len(periods) < 50:
            raise ValueError("Rejilla autoperiod insuficiente.")
    except Exception:
        # Reserva: rejilla lineal, igual que en la versión original.
        periods = np.linspace(PERIODO_MIN_DIAS, pmax, int(n_frecuencias))

    # oversample controla cuántos bins de fase se prueban por cada
    # duración; el valor por defecto de astropy es 10. Bajarlo a 6 reduce
    # notablemente el cómputo con una pérdida de resolución de fase
    # pequeña -las duraciones ya vienen acotadas físicamente, así que no
    # hace falta tanta finura-.
    res = bls.power(periods, durations, objective="snr", oversample=6)

    power = np.asarray(res.power, dtype=float)
    idx = int(np.nanargmax(power))

    best_period = float(res.period[idx])
    best_duration = float(res.duration[idx])
    depth = abs(a_float(res.depth[idx]))
    depth_err = abs(a_float(res.depth_err[idx]))
    snr = depth / depth_err if np.isfinite(depth_err) and depth_err > 0 else np.nan

    best_t0 = float(res.transit_time[idx])

    # SDE (Signal Detection Efficiency, Kovács et al. 2002): compara el
    # pico máximo del periodograma con la dispersión robusta (MAD) del
    # resto del periodograma. Complementa al SNR de BLS -que solo mira
    # depth/depth_err en el pico- con una medida de qué tan "sobresaliente"
    # es ese pico frente al ruido de fondo del propio periodograma; ayuda a
    # detectar falsos positivos con SNR nominal alto pero cuyo pico no
    # destaca realmente del resto de períodos probados.
    mediana_p = np.nanmedian(power)
    mad_p = np.nanmedian(np.abs(power - mediana_p)) * 1.4826
    sde = (power[idx] - mediana_p) / mad_p if mad_p > 0 else np.nan

    return {
        "periods": periods,
        "power": power,
        "best_period": best_period,
        "best_t0": best_t0,
        "best_duration": best_duration,
        "depth": depth,
        "depth_err": depth_err,
        "snr": snr,
        "sde": float(sde) if np.isfinite(sde) else np.nan
    }


def enmascarar_transito(time_days, period, duration):
    phase = ((time_days - time_days[0] + 0.5 * period) % period) - 0.5 * period
    return np.abs(phase) > (duration / 2.0)


def buscar_segunda_senal(lc, primera, n_frecuencias=6000, rstar_rsun=np.nan, mstar_msun=np.nan):
    time_days = np.asarray(lc.time.value, dtype=float)
    flux = np.asarray(lc.flux.value, dtype=float)

    keep = enmascarar_transito(
        time_days,
        primera["best_period"],
        primera["best_duration"]
    )

    if keep.sum() < 300:
        return {"segunda_senal": False, "periodo": np.nan, "snr": np.nan}

    lc2 = lk.LightCurve(
        time=time_days[keep] * u.day,
        flux=flux[keep]
    )

    try:
        # La segunda pasada es una verificación adicional (¿hay otro
        # planeta?), no el resultado principal: usar la mitad de
        # frecuencias es suficiente para detectar una señal razonablemente
        # fuerte y corta el tiempo de esta segunda búsqueda a la mitad.
        segunda = ejecutar_bls(
            lc2,
            n_frecuencias=max(1000, int(n_frecuencias) // 2),
            rstar_rsun=rstar_rsun,
            mstar_msun=mstar_msun
        )
    except Exception:
        return {"segunda_senal": False, "periodo": np.nan, "snr": np.nan}

    p1 = primera["best_period"]
    p2 = segunda["best_period"]

    relaciones = [0.5, 1/3, 2/3, 2.0, 3.0, 1.0]
    es_armonico = any(abs(p2 / p1 - r) < 0.03 for r in relaciones)
    es_instrumental = _es_periodo_instrumental(p2)

    snr2 = segunda["snr"]
    segunda_ok = bool(
        np.isfinite(snr2)
        and snr2 >= SNR_MEDIO
        and not es_armonico
        and not es_instrumental
    )

    return {
        "segunda_senal": segunda_ok,
        "periodo": p2,
        "snr": snr2
    }


def probar_odd_even(time_days, flux, best_t0, period, duration):
    """Test odd-even: compara la profundidad del tránsito medida por
    separado en ciclos pares e impares desde t0. Una diferencia grande
    entre ambas es la firma clásica de una binaria eclipsante casi de
    canto en la que BLS se ha "enganchado" al doble del período real (el
    eclipse secundario, algo menos profundo, queda a mitad de camino y se
    confunde con tránsitos alternos de profundidad distinta). Un planeta
    real no debería mostrar esta diferencia.

    Devuelve (diferencia_relativa, es_sospechoso).
    """
    if not (np.isfinite(period) and period > 0 and np.isfinite(duration) and duration > 0):
        return np.nan, False

    ciclo = np.floor((time_days - best_t0) / period + 0.5)
    es_par = (np.mod(ciclo, 2) == 0)

    fase = ((time_days - best_t0 + 0.5 * period) % period) - 0.5 * period
    en_transito = np.abs(fase) < (duration / 2.0)

    flujo_par = flux[en_transito & es_par]
    flujo_impar = flux[en_transito & ~es_par]

    if len(flujo_par) < 5 or len(flujo_impar) < 5:
        return np.nan, False

    prof_par = 1.0 - float(np.nanmedian(flujo_par))
    prof_impar = 1.0 - float(np.nanmedian(flujo_impar))

    if not (np.isfinite(prof_par) and np.isfinite(prof_impar)):
        return np.nan, False

    prof_media = (abs(prof_par) + abs(prof_impar)) / 2.0
    if prof_media <= 0:
        return np.nan, False

    diferencia_relativa = abs(prof_par - prof_impar) / prof_media
    # Umbral conservador: solo se marca sospechoso si la diferencia es
    # grande (>50%) Y la profundidad media es lo bastante grande como para
    # que la diferencia no sea pura estadística de bajo número de puntos.
    sospechoso = bool(diferencia_relativa > 0.5 and prof_media > 0.0005)

    return float(diferencia_relativa), sospechoso


def buscar_eclipse_secundario(time_days, flux, best_t0, period, duration):
    """Busca un mínimo secundario a mitad de camino entre dos tránsitos
    (fase orbital 0.5). Un eclipse secundario claramente significativo ahí
    es evidencia fuerte de una binaria eclipsante y no de un planeta
    (salvo el caso raro de eclipses térmicos de "hot Jupiters", mucho más
    tenues que esto). Devuelve (profundidad_secundaria, hay_eclipse)."""
    if not (np.isfinite(period) and period > 0 and np.isfinite(duration) and duration > 0):
        return np.nan, False

    fase = ((time_days - best_t0 + 0.5 * period) % period) - 0.5 * period
    en_transito = np.abs(fase) < (duration / 2.0)
    en_ventana_secundaria = np.abs(np.abs(fase) - period / 2.0) < (duration / 2.0)

    fuera = ~en_transito & ~en_ventana_secundaria

    if en_ventana_secundaria.sum() < 5 or fuera.sum() < 20:
        return np.nan, False

    nivel_base = np.nanmedian(flux[fuera])
    nivel_secundario = np.nanmedian(flux[en_ventana_secundaria])
    ruido_local = np.nanstd(flux[fuera])

    if not (np.isfinite(nivel_base) and np.isfinite(ruido_local)) or ruido_local <= 0:
        return np.nan, False

    profundidad_secundaria = float(nivel_base - nivel_secundario)
    n_sec = int(en_ventana_secundaria.sum())
    error_estandar = ruido_local / math.sqrt(max(1, n_sec))
    significancia = profundidad_secundaria / error_estandar if error_estandar > 0 else np.nan

    hay_eclipse = bool(
        np.isfinite(significancia)
        and profundidad_secundaria > 0
        and significancia > 4.0
    )

    return profundidad_secundaria, hay_eclipse


def clasificar(snr, depth, sde=np.nan, es_binaria_probable=False):
    """Clasifica la confianza de la señal (BAJA/MEDIA/ALTA).

    Además del SNR y la profundidad ya usados en la versión original,
    ahora también:
    - exige un SDE mínimo cuando está disponible, para no premiar con
      confianza alta un pico que en términos absolutos tiene buen SNR
      pero que no se distingue realmente del resto del periodograma;
    - degrada automáticamente a BAJA si el test odd-even o el eclipse
      secundario apuntan a que probablemente es una binaria eclipsante y
      no un tránsito planetario.
    """
    if es_binaria_probable:
        return "BAJA"

    if not np.isfinite(snr):
        return "BAJA"

    sde_ok_alta = (not np.isfinite(sde)) or sde >= 6.0
    sde_ok_media = (not np.isfinite(sde)) or sde >= 4.0

    if snr >= SNR_ALTO and 0.0001 <= depth <= 0.08 and sde_ok_alta:
        return "ALTA"
    if snr >= SNR_MEDIO and 0.00005 <= depth <= 0.12 and sde_ok_media:
        return "MEDIA"
    return "BAJA"


def propiedades_planeta(depth, rstar_rsun, mstar_msun, period_days, teff):
    radio_rearth = np.nan
    radio_rjup = np.nan
    semieje_au = np.nan
    teq = np.nan

    if np.isfinite(depth) and depth > 0 and np.isfinite(rstar_rsun) and rstar_rsun > 0:
        rp_rstar = math.sqrt(depth)
        rp_rsun = rp_rstar * rstar_rsun
        radio_rearth = (rp_rsun * R_sun).to(u.R_earth).value
        radio_rjup = (rp_rsun * R_sun).to(u.R_jup).value

    if np.isfinite(period_days) and np.isfinite(mstar_msun) and mstar_msun > 0:
        P = period_days * u.day
        M = mstar_msun * M_sun
        a = ((G * M * P**2 / (4 * np.pi**2)) ** (1/3)).to(u.AU)
        semieje_au = float(a.value)

        if np.isfinite(teff) and np.isfinite(rstar_rsun) and rstar_rsun > 0:
            rs_au = (rstar_rsun * R_sun).to(u.AU).value
            teq = float(teff * math.sqrt(rs_au / (2 * semieje_au)))

    return radio_rearth, radio_rjup, semieje_au, teq


def consultar_exoplanet_archive(tic_id):
    """Busca planetas confirmados asociados al TIC en PSCompPars."""
    try:
        tap = TapPlus(url="https://exoplanetarchive.ipac.caltech.edu/TAP")
        query = f"""
        SELECT pl_name, hostname, tic_id, pl_orbper, pl_rade, pl_radj, pl_bmassj, pl_dens
        FROM pscomppars
        WHERE tic_id = {int(tic_id)}
        """
        job = tap.launch_job(query)
        table = job.get_results()

        if len(table) == 0:
            return {"confirmado": False, "planetas": [], "tabla": table}

        nombres = [str(x) for x in table["pl_name"]]
        return {"confirmado": True, "planetas": nombres, "tabla": table}

    except Exception as e:
        print("Aviso: no se pudo consultar Exoplanet Archive:", e)
        return {"confirmado": False, "planetas": [], "tabla": None}


def comparar_periodo(periodo_detectado, archive_info):
    if not archive_info["confirmado"] or archive_info["tabla"] is None:
        return np.nan, np.nan

    vals = []
    for x in archive_info["tabla"]["pl_orbper"]:
        v = a_float(x)
        if np.isfinite(v) and v > 0:
            vals.append(v)

    if not vals:
        return np.nan, np.nan

    vals = np.array(vals, dtype=float)
    idx = int(np.argmin(np.abs(vals - periodo_detectado)))
    publicado = float(vals[idx])
    error_pct = abs(periodo_detectado - publicado) / publicado * 100

    return publicado, error_pct


def densidad_planeta_si_hay_masa(radio_reearth, archive_info, periodo_detectado):
    if not np.isfinite(radio_reearth) or radio_reearth <= 0:
        return np.nan

    if not archive_info["confirmado"] or archive_info["tabla"] is None:
        return np.nan

    tabla = archive_info["tabla"]
    candidatos = []

    for i in range(len(tabla)):
        p = a_float(tabla["pl_orbper"][i])
        m = a_float(tabla["pl_bmassj"][i])

        if np.isfinite(p) and np.isfinite(m) and m > 0:
            candidatos.append((abs(p - periodo_detectado), m))

    if not candidatos:
        return np.nan

    candidatos.sort(key=lambda x: x[0])
    _, masa_mj = candidatos[0]

    masa_kg = (masa_mj * u.M_jup).to(u.kg).value
    radio_m = (radio_reearth * u.R_earth).to(u.m).value
    volumen = 4 / 3 * np.pi * radio_m**3
    densidad = masa_kg / volumen

    return float(densidad / 1000.0)


def extraer_sectores(search):
    for col in ["sequence_number", "sector"]:
        try:
            if col in search.table.colnames:
                vals = sorted(set(str(x) for x in search.table[col]))
                return ",".join(vals) if vals else "No disponible"
        except Exception:
            pass
    return "No disponible"


def fechas_curva(lc):
    try:
        return lc.time[0].isot, lc.time[-1].isot
    except Exception:
        return str(lc.time[0]), str(lc.time[-1])


def _submuestrear_para_grafico(*arrays, limite=8000):
    """Reduce arrays a lo sumo a `limite` puntos, tomados a intervalos
    regulares, SOLO para dibujarlos. El análisis (BLS, profundidades,
    odd-even, etc.) ya se hizo antes sobre la curva completa; esto no
    cambia ningún resultado, solo lo que se pinta. Dibujar decenas de
    miles de puntos como scatter en cada figura -una por estrella,
    potencialmente cientos en modo automático- es lento y le exige mucha
    memoria al renderizador de matplotlib sin aportar nada visualmente que
    ya no se vea con un subconjunto representativo."""
    n = len(arrays[0])
    if n <= limite:
        return arrays if len(arrays) > 1 else arrays[0]

    idx = np.linspace(0, n - 1, limite).astype(int)
    reducidos = tuple(np.asarray(a)[idx] for a in arrays)
    return reducidos if len(reducidos) > 1 else reducidos[0]


def generar_figura(lc, bls_info, tic_id, guardado_en, nombre_estrella=None,
                    diagnostico_extra=None, mostrar=True):
    """Figura estilo 'observatorio' (tema oscuro) con 4 paneles: curva
    completa con tránsitos marcados, zoom al primer tránsito, periodograma
    BLS con armónicos (ahora también con el valor de SDE) y curva plegada
    binada. `diagnostico_extra`, si se pasa, añade una pequeña anotación
    con el resultado del test odd-even y del eclipse secundario.

    `mostrar=False` guarda el PNG en disco pero NO la muestra en línea en
    el notebook. Mostrarla (plt.show()) la incrusta como imagen en la
    salida de la celda; en el modo automático, que procesa estrellas sin
    parar, eso hace crecer sin límite la memoria del kernel y del
    navegador. En el modo manual (un análisis a la vez) sí se muestra,
    como antes.
    """
    time_days = np.asarray(lc.time.value, dtype=float)
    flux = np.asarray(lc.flux.value, dtype=float)

    p = bls_info["best_period"]
    dur = bls_info["best_duration"]
    t0 = bls_info.get("best_t0", time_days[0])
    etiqueta = nombre_estrella or f"TIC {tic_id}"

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.patch.set_facecolor(PALETTE["bg"])
    for ax in axes.flat:
        ax.set_facecolor(PALETTE["bg"])
    fig.suptitle(f"{etiqueta} — Pipeline de detección TESS",
                 fontsize=14, fontweight="bold", color=PALETTE["white"], y=0.98)

    # ---------- Panel 1: curva de luz completa, tránsitos marcados ----------
    ax1 = axes[0, 0]
    t_plot1, f_plot1 = _submuestrear_para_grafico(time_days, flux, limite=8000)
    ax1.plot(t_plot1, f_plot1, ".", ms=1.5, alpha=0.5, color=PALETTE["blue"],
             label="Flujo (detrended)")

    t_ref = t0
    while t_ref > time_days.min():
        t_ref -= p
    primero = True
    while t_ref < time_days.max():
        if t_ref >= time_days.min():
            ax1.axvline(t_ref, color=PALETTE["red"], alpha=0.4, lw=1, linestyle="--",
                        label=f"Tránsitos (P={p:.4f} d)" if primero else None)
            primero = False
        t_ref += p

    ax1.set_title("Curva de luz completa", fontsize=11, color=PALETTE["white"])
    ax1.set_xlabel("Tiempo [BTJD]")
    ax1.set_ylabel("Flujo normalizado")
    ax1.legend(fontsize=8, loc="upper right")
    ax1.grid(True)

    # ---------- Panel 2: zoom al primer tránsito detectado ----------
    ax2 = axes[0, 1]
    ventana = max(dur * 3, 0.3)
    fase_centrada = ((time_days - t0 + 0.5 * p) % p) - 0.5 * p
    mascara_zoom = np.abs(fase_centrada) < ventana

    if mascara_zoom.sum() > 5:
        t_zoom = fase_centrada[mascara_zoom] * 24  # a horas relativas al tránsito
        f_zoom = flux[mascara_zoom]
        orden = np.argsort(t_zoom)
        ax2.plot(t_zoom[orden], f_zoom[orden], ".", ms=3, color=PALETTE["green"], label="Datos")
        ax2.axvspan(-dur / 2 * 24, dur / 2 * 24, color=PALETTE["red"], alpha=0.15,
                    label="Tránsito estimado")
    else:
        ax2.text(0.5, 0.5, "Datos insuficientes\npara zoom", ha="center", va="center",
                 color=PALETTE["muted"], transform=ax2.transAxes)

    if diagnostico_extra:
        texto_diag = diagnostico_extra.get("texto")
        if texto_diag:
            ax2.text(
                0.02, 0.02, texto_diag, transform=ax2.transAxes,
                fontsize=7.5, color=PALETTE["gold"], va="bottom", ha="left"
            )

    ax2.set_title("Zoom — tránsito individual", fontsize=11, color=PALETTE["white"])
    ax2.set_xlabel("Horas relativas al centro del tránsito")
    ax2.set_ylabel("Flujo normalizado")
    ax2.legend(fontsize=8)
    ax2.grid(True)

    # ---------- Panel 3: periodograma BLS con armónicos ----------
    ax3 = axes[1, 0]
    ax3.plot(bls_info["periods"], bls_info["power"], color=PALETTE["blue"], lw=0.8)
    ax3.axvline(p, color=PALETTE["red"], lw=2, label=f"Pico: {p:.4f} d")
    for n in (2, 3):
        if p * n <= bls_info["periods"].max():
            ax3.axvline(p * n, color=PALETTE["orange"], lw=1, linestyle=":",
                        label=f"Armónico {n}x" if n == 2 else None)

    snr_txt = f"{bls_info['snr']:.2f}" if np.isfinite(bls_info["snr"]) else "N/D"
    sde_val = bls_info.get("sde", np.nan)
    sde_txt = f"{sde_val:.2f}" if np.isfinite(sde_val) else "N/D"
    ax3.set_title(f"Periodograma BLS — SNR={snr_txt} · SDE={sde_txt}",
                  fontsize=11, color=PALETTE["white"])
    ax3.set_xlabel("Período de prueba [días]")
    ax3.set_ylabel("Potencia BLS")
    ax3.legend(fontsize=8)
    ax3.grid(True)

    # ---------- Panel 4: curva plegada en fase, binada ----------
    ax4 = axes[1, 1]
    fase = ((time_days - t0) % p) / p
    fase = np.where(fase > 0.5, fase - 1.0, fase)
    orden = np.argsort(fase)
    fase_ord, flujo_ord = fase[orden], flux[orden]

    # El scatter de fondo se submuestrea solo para dibujar (más rápido y
    # más liviano); la mediana binada de abajo se calcula con los datos
    # COMPLETOS (fase_ord/flujo_ord), así que la curva binada no pierde
    # precisión por esto.
    fase_plot, flujo_plot = _submuestrear_para_grafico(fase_ord, flujo_ord, limite=8000)
    ax4.scatter(fase_plot, flujo_plot, s=3, alpha=0.3, color=PALETTE["blue"], label="Datos")

    n_bins = 60
    bins = np.linspace(-0.5, 0.5, n_bins + 1)
    centros = (bins[:-1] + bins[1:]) / 2
    flujo_bin = np.array([
        np.nanmedian(flujo_ord[(fase_ord >= bins[i]) & (fase_ord < bins[i + 1])])
        if np.any((fase_ord >= bins[i]) & (fase_ord < bins[i + 1])) else np.nan
        for i in range(n_bins)
    ])
    ax4.plot(centros, flujo_bin, color=PALETTE["white"], lw=1.8, label="Mediana binada")
    ax4.axvline(0.5, color=PALETTE["orange"], lw=1, linestyle=":", alpha=0.6,
                label="Fase 0.5 (posible eclipse 2°)")
    ax4.axvline(-0.5, color=PALETTE["orange"], lw=1, linestyle=":", alpha=0.6)

    ax4.set_title("Curva plegada en fase", fontsize=11, color=PALETTE["white"])
    ax4.set_xlabel("Fase orbital")
    ax4.set_ylabel("Flujo normalizado")
    ax4.set_xlim(-0.5, 0.5)
    ax4.legend(fontsize=8)
    ax4.grid(True)

    plt.tight_layout()

    fig_dir = Path(guardado_en).parent / "figuras_exoplanetas"
    fig_dir.mkdir(parents=True, exist_ok=True)

    fig_path = fig_dir / f"TIC_{tic_id}.png"
    fig.savefig(fig_path, dpi=120, bbox_inches="tight", facecolor=fig.get_facecolor())

    if mostrar:
        plt.show()

    plt.close(fig)
    del fig, axes
    gc.collect()

    return fig_path


def imprimir_resumen_parametros(fila):
    """Resumen en caja de texto, con el mismo estilo visual que
    detector_exoplanetas.ipynb (Sección D: estimación de parámetros)."""
    ancho = 58

    def valor_txt(v, fmt="{:.4f}"):
        if isinstance(v, float):
            return fmt.format(v) if np.isfinite(v) else "N/D"
        return str(v) if v not in (None, "") else "N/D"

    filas = [
        ("Estrella", fila.get("nombre_estrella", "N/D")),
        ("Estado", fila.get("estado", "N/D")),
        ("Confianza", fila.get("confianza", "N/D")),
        ("Período detectado [d]", valor_txt(fila.get("periodo_detectado_dias", np.nan))),
        ("SNR (BLS)", valor_txt(fila.get("snr_bls", np.nan), "{:.2f}")),
        ("Profundidad tránsito [%]", valor_txt(fila.get("profundidad_pct", np.nan), "{:.4f}")),
        ("Posible binaria eclipsante", valor_txt(fila.get("posible_binaria_eclipsante", "N/D"))),
        ("Segunda señal (multiplaneta)", valor_txt(fila.get("posible_segunda_senal", "N/D"))),
        ("Radio planeta [R⊕]", valor_txt(fila.get("radio_planeta_Rearth", np.nan), "{:.2f}")),
        ("Radio planeta [Rj]", valor_txt(fila.get("radio_planeta_Rjup", np.nan), "{:.3f}")),
        ("Semieje mayor [AU]", valor_txt(fila.get("semieje_mayor_AU", np.nan), "{:.4f}")),
        ("Temp. equilibrio [K]", valor_txt(fila.get("temperatura_equilibrio_K", np.nan), "{:.0f}")),
        ("Densidad [g/cm³]", valor_txt(fila.get("densidad_g_cm3", np.nan), "{:.2f}")),
        ("Confirmado en Exoplanet Archive", valor_txt(fila.get("planeta_confirmado_en_archive", "N/D"))),
    ]

    print("╔" + "═" * ancho + "╗")
    titulo = f" PARÁMETROS ESTIMADOS "
    print("║" + titulo.center(ancho) + "║")
    print("╠" + "═" * ancho + "╣")
    for etiqueta, valor in filas:
        linea = f" {etiqueta:<32}: {valor}"
        print("║" + linea[:ancho].ljust(ancho) + "║")
    print("╚" + "═" * ancho + "╝")


def procesar_una_estrella(carpeta_salida, nombre_csv, nombre_historial, n_frecuencias,
                           mostrar_figura=True):
    carpeta = Path(carpeta_salida).expanduser().resolve()
    carpeta.mkdir(parents=True, exist_ok=True)

    csv_resultados = carpeta / nombre_csv
    csv_historial = carpeta / nombre_historial

    ids_usados = cargar_ids_usados(csv_historial)

    print("Buscando una estrella TESS nueva...")
    row = buscar_estrellas_nuevas(ids_usados)
    datos = obtener_datos_tic(row)

    tic_id = datos["tic_id"]
    print(f"Seleccionada: TIC {tic_id} | Tmag={datos['tmag']:.2f}")

    # Mismas 35 columnas, mismos valores por defecto que la versión
    # original: el formato del CSV no cambia.
    fila = {
        **datos,
        "sectores_tess": "",
        "observacion_tess_inicio": "",
        "observacion_tess_fin": "",
        "fecha_hora_analisis": datetime.now().astimezone().isoformat(),
        "datos_tess_disponibles": False,
        "num_puntos_curva": 0,
        "periodo_detectado_dias": np.nan,
        "snr_bls": np.nan,
        "profundidad_fraccion": np.nan,
        "profundidad_pct": np.nan,
        "confianza": "BAJA",
        "anomalia_en_la_luz": False,
        "posible_transito": False,
        "posible_binaria_eclipsante": False,
        "posible_segunda_senal": False,
        "periodo_segunda_senal_dias": np.nan,
        "snr_segunda_senal": np.nan,
        "planeta_confirmado_en_archive": False,
        "planetas_conocidos": "",
        "periodo_publicado_mas_cercano_dias": np.nan,
        "error_periodo_porcentual": np.nan,
        "radio_planeta_Rearth": np.nan,
        "radio_planeta_Rjup": np.nan,
        "semieje_mayor_AU": np.nan,
        "temperatura_equilibrio_K": np.nan,
        "densidad_g_cm3": np.nan,
        "estado": "SIN DATOS",
        "mensaje": ""
    }

    try:
        lc, search = descargar_curva_tess(tic_id)

        if lc is None:
            fila["estado"] = "SIN CURVA TESS"
            fila["mensaje"] = "El TIC no tuvo curvas de luz TESS descargables."

        else:
            fila["datos_tess_disponibles"] = True
            fila["sectores_tess"] = extraer_sectores(search)

            lc = preparar_curva(lc)
            fila["num_puntos_curva"] = int(len(lc))

            inicio, fin = fechas_curva(lc)
            fila["observacion_tess_inicio"] = inicio
            fila["observacion_tess_fin"] = fin

            rstar = datos["radio_estrella_Rsun"]
            mstar = datos["masa_estrella_Msun"]

            bls_info = ejecutar_bls(
                lc,
                n_frecuencias=n_frecuencias,
                rstar_rsun=rstar,
                mstar_msun=mstar
            )

            depth = bls_info["depth"]
            snr = bls_info["snr"]
            periodo = bls_info["best_period"]

            time_days = np.asarray(lc.time.value, dtype=float)
            flux = np.asarray(lc.flux.value, dtype=float)

            # --- Diagnóstico de binaria eclipsante: odd-even + eclipse 2° ---
            odd_even_diff, odd_even_sospechoso = probar_odd_even(
                time_days, flux, bls_info["best_t0"], periodo, bls_info["best_duration"]
            )
            prof_secundaria, hay_secundario = buscar_eclipse_secundario(
                time_days, flux, bls_info["best_t0"], periodo, bls_info["best_duration"]
            )
            periodo_instrumental = _es_periodo_instrumental(periodo)

            fila["periodo_detectado_dias"] = periodo
            fila["snr_bls"] = snr
            fila["profundidad_fraccion"] = depth
            fila["profundidad_pct"] = depth * 100

            fila["posible_binaria_eclipsante"] = bool(
                depth >= 0.03 or odd_even_sospechoso or hay_secundario
            )

            fila["confianza"] = clasificar(
                snr, depth,
                sde=bls_info.get("sde", np.nan),
                es_binaria_probable=fila["posible_binaria_eclipsante"]
            )

            fila["anomalia_en_la_luz"] = bool(
                np.isfinite(snr) and snr >= SNR_BAJO
            )
            fila["posible_transito"] = bool(
                np.isfinite(snr)
                and snr >= SNR_MEDIO
                and 0.00005 <= depth <= 0.12
                and not periodo_instrumental
                and not fila["posible_binaria_eclipsante"]
            )

            if periodo_instrumental:
                print(f"⚠️ Período detectado ({periodo:.4f} d) muy cercano a un alias "
                      f"instrumental típico de TESS: se descarta como candidato de tránsito.")

            if fila["posible_binaria_eclipsante"]:
                motivos = []
                if depth >= 0.03:
                    motivos.append(f"profundidad grande ({depth*100:.2f}%)")
                if odd_even_sospechoso:
                    motivos.append(f"diferencia odd-even={odd_even_diff:.2f}")
                if hay_secundario:
                    motivos.append("eclipse secundario detectado en fase 0.5")
                print("⚠️ Posible binaria eclipsante —", "; ".join(motivos))

            segunda = buscar_segunda_senal(
                lc,
                bls_info,
                n_frecuencias=n_frecuencias,
                rstar_rsun=rstar,
                mstar_msun=mstar
            )

            fila["posible_segunda_senal"] = segunda["segunda_senal"]
            fila["periodo_segunda_senal_dias"] = segunda["periodo"]
            fila["snr_segunda_senal"] = segunda["snr"]

            archive = consultar_exoplanet_archive(tic_id)

            fila["planeta_confirmado_en_archive"] = archive["confirmado"]
            fila["planetas_conocidos"] = "; ".join(archive["planetas"])

            publicado, error_pct = comparar_periodo(
                periodo,
                archive
            )
            fila["periodo_publicado_mas_cercano_dias"] = publicado
            fila["error_periodo_porcentual"] = error_pct

            rp_e, rp_j, a_au, teq = propiedades_planeta(
                depth,
                rstar,
                mstar,
                periodo,
                datos["teff_K"]
            )

            fila["radio_planeta_Rearth"] = rp_e
            fila["radio_planeta_Rjup"] = rp_j
            fila["semieje_mayor_AU"] = a_au
            fila["temperatura_equilibrio_K"] = teq
            fila["densidad_g_cm3"] = densidad_planeta_si_hay_masa(
                rp_e,
                archive,
                periodo
            )

            fila["estado"] = "ANALIZADO"
            fila["mensaje"] = (
                "Candidato BLS detectado."
                if fila["posible_transito"]
                else "No se obtuvo una señal de tránsito suficientemente fuerte."
            )

            texto_diag = None
            if np.isfinite(odd_even_diff):
                texto_diag = f"odd-even Δ={odd_even_diff:.2f}"
            if np.isfinite(prof_secundaria):
                texto_diag = (texto_diag + "  " if texto_diag else "") + \
                    f"ecl.2°={prof_secundaria*100:.3f}%"

            fig_path = generar_figura(
                lc,
                bls_info,
                tic_id,
                csv_resultados,
                nombre_estrella=datos["nombre_estrella"],
                diagnostico_extra={"texto": texto_diag} if texto_diag else None,
                mostrar=mostrar_figura
            )
            fila["mensaje"] += f" Figura: {fig_path.name}"

            # Libera explícitamente los arreglos grandes de esta estrella
            # (la curva de luz completa, tiempo y flujo) en vez de esperar
            # a que Python los recolecte cuando le convenga. En una sesión
            # de modo automático que procesa estrellas sin parar, esto
            # evita que la memoria vaya creciendo entre una estrella y la
            # siguiente mientras el recolector de basura no se dispara.
            del lc, time_days, flux, bls_info

    except Exception as e:
        fila["estado"] = "ERROR"
        fila["mensaje"] = repr(e)

    append_resultado(csv_resultados, fila)
    append_historial(csv_historial, fila)

    gc.collect()

    return fila, csv_resultados, csv_historial


def mostrar_descarga(path):
    path = Path(path).resolve()
    print(f"CSV guardado en: {path}")

    try:
        from google.colab import files
        files.download(str(path))
        return
    except Exception:
        pass

    try:
        display(FileLink(str(path), result_html_prefix="Descargar CSV: "))
    except Exception:
        pass


def ejecutar_desde_interfaz(_=None):
    with salida_ui:
        clear_output(wait=True)

        carpeta = ruta_widget.value.strip()
        nombre_csv = nombre_widget.value.strip()
        nombre_historial = historial_widget.value.strip()
        nfreq = int(frecuencias_widget.value)

        if not carpeta:
            print("Escribe una carpeta de salida.")
            return

        if not nombre_csv.lower().endswith(".csv"):
            nombre_csv += ".csv"

        if not nombre_historial.lower().endswith(".csv"):
            nombre_historial += ".csv"

        try:
            fila, csv_path, hist_path = procesar_una_estrella(
                carpeta,
                nombre_csv,
                nombre_historial,
                nfreq
            )

            print()
            imprimir_resumen_parametros(fila)
            print(f"\nPlanetas conocidos: {fila['planetas_conocidos'] or 'ninguno registrado'}")
            print(f"Mensaje: {fila['mensaje']}")

            print("\n===== ARCHIVOS =====")
            mostrar_descarga(csv_path)
            print("Historial:", hist_path)

        except Exception as e:
            print("ERROR:", repr(e))


analizar_btn.on_click(ejecutar_desde_interfaz)


## Cómo usarlo

Escribe la **carpeta** donde quieres guardar los resultados y el **nombre del CSV**. Luego pulsa **ANALIZAR UNA ESTRELLA**.

Cada pulsación:

- elige un TIC aleatorio
- evita los TIC que ya estén en `historial_exoplanetas.csv`
- intenta descargar sus datos TESS
- ejecuta BLS con el número de frecuencias indicado
- añade una nueva fila al CSV
- genera una figura PNG en `figuras_exoplanetas`
- muestra un enlace para descargar el CSV

El historial separado es intencional: así puedes cambiar el nombre del CSV de resultados sin volver a analizar accidentalmente una estrella anterior.

### Campos importantes del CSV

`fecha_hora_analisis` registra cuándo se ejecutó el análisis.  
`observacion_tess_inicio` y `observacion_tess_fin` registran el intervalo temporal de los datos TESS utilizados.  
`anomalia_en_la_luz`, `posible_transito` y `posible_binaria_eclipsante` son banderas heurísticas.  
`planeta_confirmado_en_archive`, `planetas_conocidos` y `error_periodo_porcentual` permiten comparar el resultado con NASA Exoplanet Archive.

La detección automática genera **candidatos**, no confirmaciones. Una señal periódica puede proceder de binarias eclipsantes, variabilidad estelar, sistemáticas instrumentales u otros falsos positivos.


# Modo automático continuo

Ejecuta la celda siguiente y el programa comenzará a analizar estrellas una detrás de otra automáticamente.

No necesitas pulsar ningún botón. Cada resultado se añade al CSV y el historial evita repetir TIC ya analizados.

Para detenerlo en Jupyter usa **Kernel → Interrupt Kernel** o el botón de interrupción. Puedes cambiar `PAUSA_ENTRE_ESTRELLAS` antes de iniciar el bucle.


In [ ]:
# ============================================================
# MODO AUTOMÁTICO CONTINUO
# ============================================================
#
# Nota sobre consumo de RAM: en versiones anteriores, cada estrella
# analizada dejaba su imagen y sus textos apilados en la salida de esta
# celda, sin borrar nunca los de la estrella anterior. En una sesión que
# corre indefinidamente eso hace crecer la memoria del kernel (y del
# navegador, que tiene que seguir guardando cada imagen incrustada) sin
# límite, hasta agotar la RAM y empezar a usar swap sin parar. Ahora cada
# vuelta del bucle limpia la salida de la anterior (clear_output) y las
# figuras solo se guardan en disco, no se muestran en línea -se pueden
# abrir después desde la carpeta 'figuras_exoplanetas'-.

import time
import gc
from IPython.display import clear_output

PAUSA_ENTRE_ESTRELLAS = 5
PAUSA_MAXIMA_TRAS_ERRORES = 120
CONTINUAR_SI_HAY_ERROR = True
ESTRELLAS_ANALIZADAS_EN_ESTA_SESION = 0


def ejecutar_modo_automatico():
    global ESTRELLAS_ANALIZADAS_EN_ESTA_SESION

    carpeta = ruta_widget.value.strip() or os.getcwd()
    nombre_csv = nombre_widget.value.strip() or "resultados_exoplanetas.csv"
    nombre_historial = (
        historial_widget.value.strip()
        or "historial_estrellas_exoplanetas.csv"
    )
    nfreq = int(frecuencias_widget.value)

    if not nombre_csv.lower().endswith(".csv"):
        nombre_csv += ".csv"
    if not nombre_historial.lower().endswith(".csv"):
        nombre_historial += ".csv"

    ruta_absoluta = Path(carpeta).expanduser().resolve()

    # Cuenta errores consecutivos para aplicar un backoff exponencial
    # acotado (ver más abajo). Se reinicia a 0 en cuanto una estrella se
    # procesa sin excepción, aunque el resultado sea "SIN CURVA TESS" o
    # similar (eso no es un error del pipeline, es un resultado válido).
    errores_consecutivos = 0

    while True:
        numero = ESTRELLAS_ANALIZADAS_EN_ESTA_SESION + 1
        hora = datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S")

        # Se limpia la salida de la vuelta anterior ANTES de empezar la
        # nueva: así el notebook (y la memoria del navegador/kernel que
        # la sostiene) solo carga con el texto de la última estrella, en
        # vez de acumular sin límite el de todas las que se han
        # procesado desde que arrancó el modo automático.
        clear_output(wait=True)

        print("=" * 70)
        print("🔭 DETECTOR DE EXOPLANETAS — MODO AUTOMÁTICO CONTINUO")
        print("=" * 70)
        print("Carpeta:", ruta_absoluta)
        print("CSV:", nombre_csv, "| Historial:", nombre_historial)
        print("Frecuencias BLS:", nfreq)
        print("Analizadas en esta sesión:", ESTRELLAS_ANALIZADAS_EN_ESTA_SESION)
        print("Para detenerlo: Kernel → Interrupt Kernel")
        print("\n" + "=" * 70)
        print(f"▶ ESTRELLA #{numero} — {hora}")
        print("=" * 70)

        try:
            fila, csv_path, hist_path = procesar_una_estrella(
                carpeta,
                nombre_csv,
                nombre_historial,
                nfreq,
                mostrar_figura=False
            )

            ESTRELLAS_ANALIZADAS_EN_ESTA_SESION += 1
            errores_consecutivos = 0

            estado_icono = "✅" if fila["estado"] == "ANALIZADO" else "⚠️"
            print(f"\n{estado_icono} Análisis terminado")
            imprimir_resumen_parametros(fila)
            print(f"\nCSV actualizado: {csv_path}")
            print(f"Analizadas en esta sesión: {ESTRELLAS_ANALIZADAS_EN_ESTA_SESION}")

        except KeyboardInterrupt:
            print("\n⏹ Análisis detenido por el usuario.")
            print(
                "Estrellas analizadas en esta sesión:",
                ESTRELLAS_ANALIZADAS_EN_ESTA_SESION
            )
            break

        except Exception as e:
            errores_consecutivos += 1
            print("\n❌ ERROR EN ESTA ESTRELLA:", repr(e))
            if not CONTINUAR_SI_HAY_ERROR:
                raise
            print(f"Se continuará con otra estrella "
                  f"(errores consecutivos: {errores_consecutivos}).")

        # Backoff exponencial acotado tras errores consecutivos: si MAST,
        # SIMBAD o la red están fallando de forma repetida, espaciar los
        # reintentos evita machacar esos servicios y da tiempo a que un
        # problema transitorio se resuelva solo, en vez de repetir el
        # mismo fallo cada pocos segundos indefinidamente.
        pausa = PAUSA_ENTRE_ESTRELLAS
        if errores_consecutivos > 0:
            pausa = min(
                PAUSA_MAXIMA_TRAS_ERRORES,
                PAUSA_ENTRE_ESTRELLAS * (2 ** (errores_consecutivos - 1))
            )

        print(f"\n⏳ Esperando {pausa} segundos...")

        try:
            time.sleep(pausa)
        except KeyboardInterrupt:
            print("\nAnálisis detenido por el usuario.")
            break


# Al ejecutar esta celda comienza automáticamente.
ejecutar_modo_automatico()


🔭 DETECTOR DE EXOPLANETAS — MODO AUTOMÁTICO CONTINUO
Carpeta: /home/gael/Documents/CosasEscuela/ProyectoCiencias
CSV: resultados_exoplanetas.csv | Historial: historial_estrellas_exoplanetas.csv
Frecuencias BLS: 3000
Analizadas en esta sesión: 0
Para detenerlo: Kernel → Interrupt Kernel

▶ ESTRELLA #1 — 2026-10-02 22:31:56
Buscando una estrella TESS nueva...
Seleccionada: TIC 283994476 | Tmag=12.00


---
## Conclusiones y próximos pasos

### Lo que este pipeline automatiza:

| Etapa | Qué hace | Salida |
|---|---|---|
| Selección de TIC | Elige estrellas aleatorias sin repetir | Historial CSV |
| Descarga TESS | `lightkurve` + detrending Savitzky-Golay | Curva de luz limpia |
| BLS | Búsqueda de período sobre miles de frecuencias | Período, profundidad, SNR |
| Segunda señal | Enmascara el primer tránsito y repite BLS | Indicio de multiplaneta |
| NASA Exoplanet Archive | Consulta TAP por `tic_id` | Comparación con planetas confirmados |
| Parámetros físicos | 3ª ley de Kepler, equilibrio radiativo | Radio, semieje, T_eq, densidad |
| Figura de 4 paneles | Curva completa, zoom, periodograma, fase plegada | PNG por estrella |

### Extensiones posibles:
1. **Ajuste de modelo completo** con `batman` (Mandel & Agol 2002) en vez del modelo caja de BLS.
2. **Machine learning** para clasificar candidatos vs. falsos positivos (binarias, sistemáticas).
3. **Múltiples sectores TESS** por estrella para mejorar la precisión del período.
4. Convertir el notebook a script con `nbconvert` y programarlo con `cron` o `systemd` para que corra en un servidor sin supervisión.
5. Ampliar la clasificación con las mismas heurísticas de `TargetSelector` / `Clasificador` del pipeline modular.

### Fuentes de datos públicos usadas:
- **MAST** (Mikulski Archive for Space Telescopes): `mast.stsci.edu`
- **NASA Exoplanet Archive**: `exoplanetarchive.ipac.caltech.edu`

---
*Desarrollado con Python 3 · lightkurve · astropy · astroquery · scipy · pandas · matplotlib*